# BitoGuard AML — 一鍵執行
**點選上方 `Cell > Run All` 即可完整執行**

執行順序：
1. 安裝套件
2. 從 API 下載資料（若本地已有則跳過）
3. 特徵工程 + LightGBM 訓練
4. 輸出結果 + 顯示評估指標
5. 啟動 Streamlit 儀表板（可選）

In [ ]:
# ── Cell 1：安裝套件（conda 優先，失敗自動 fallback 到 pip）──────────────────
import importlib.util, subprocess, sys

def installed(mod):
    return importlib.util.find_spec(mod) is not None

def run(cmd, label=''):
    print(f'  執行: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ⚠ {label} 失敗：{result.stderr[-200:]}')
        return False
    return True

# ── 1. lightgbm ──────────────────────────────────────────────────────────────
if not installed('lightgbm'):
    print('[1/3] 安裝 lightgbm...')
    # 先試 conda（最快，有預編譯 binary）
    ok = run(['conda', 'install', '-c', 'conda-forge', 'lightgbm', '-y', '-q'], 'conda')
    if not ok or not installed('lightgbm'):
        # fallback: pip wheel
        print('  conda 失敗，改用 pip...')
        run([sys.executable, '-m', 'pip', 'install', '-q',
             '--prefer-binary', 'lightgbm'], 'pip lightgbm')
    if installed('lightgbm'):
        import lightgbm; print(f'  lightgbm {lightgbm.__version__} OK')
    else:
        raise RuntimeError('lightgbm 安裝失敗，請手動執行：pip install lightgbm')
else:
    import lightgbm; print(f'[1/3] lightgbm {lightgbm.__version__} 已安裝')

# ── 2. plotly ─────────────────────────────────────────────────────────────────
if not installed('plotly'):
    print('[2/3] 安裝 plotly...')
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', 'plotly'], 'pip plotly')
    if installed('plotly'):
        import plotly; print(f'  plotly {plotly.__version__} OK')
else:
    import plotly; print(f'[2/3] plotly {plotly.__version__} 已安裝')

# ── 3. streamlit ──────────────────────────────────────────────────────────────
if not installed('streamlit'):
    print('[3/3] 安裝 streamlit...')
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', 'streamlit'], 'pip streamlit')
    if installed('streamlit'):
        print('  streamlit OK')
else:
    print('[3/3] streamlit 已安裝')

print('\n全部套件就緒 OK')

In [ ]:
# ── Cell 2：從 API 下載資料（已有則跳過）──────────────────────────────────────
import urllib.request, json, time
from pathlib import Path
import pandas as pd

BASE    = Path('/home/ec2-user/SageMaker')
DATA    = BASE / 'data'
API     = 'https://aws-event-api.bitopro.com'
DATE    = '2026-03-26'
TABLES  = ['user_info','twd_transfer','crypto_transfer',
           'usdt_twd_trading','train_label','predict_label']

def fetch_table(name):
    out = DATA / name / f'dt={DATE}' / 'part-00000.csv'
    if out.exists():
        print(f'  {name}: 已存在，跳過')
        return
    out.parent.mkdir(parents=True, exist_ok=True)
    rows, offset, limit = [], 0, 1000
    print(f'  {name}: 下載中', end='', flush=True)
    while True:
        url = f'{API}/{name}?limit={limit}&offset={offset}'
        req = urllib.request.Request(url, headers={'Accept':'application/json'})
        with urllib.request.urlopen(req, timeout=30) as r:
            batch = json.loads(r.read())
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        print('.', end='', flush=True)
        if len(batch) < limit:
            break
        time.sleep(0.05)
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f' {len(rows):,} 筆 OK')

print('資料下載（已有的表格自動跳過）：')
for t in TABLES:
    fetch_table(t)
print('完成！')

In [ ]:
# ── Cell 3：特徵工程 + 訓練（Fold-Aware 圖特徵，修正 kyc_level）────────────
import warnings, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from pathlib import Path
warnings.filterwarnings('ignore')

BASE     = Path('/home/ec2-user/SageMaker')
DATA_DIR = BASE / 'data'

def load_csv(table):
    for p in sorted((DATA_DIR / table).glob('dt=*/part-*.csv'), reverse=True):
        return pd.read_csv(p, low_memory=False)
    raise FileNotFoundError(f'找不到：{table}')

# ── 圖特徵計算函數（fold-aware：只用指定的 blacklist_set）──────────────────────
def compute_graph_features(all_user_ids, cs, twd, blacklist_set):
    neighbor_count, direct_neighbor = {}, {}

    # 來源①：crypto relation_user_id（直接交易對手，無向）
    if 'relation_user_id' in cs.columns:
        edges = cs[cs['relation_user_id'].notna()][['user_id','relation_user_id']].copy()
        edges['relation_user_id'] = edges['relation_user_id'].astype(float).astype('Int64')
        edges = edges.dropna(subset=['relation_user_id'])
        for _, row in edges.iterrows():
            u, v = int(row['user_id']), int(row['relation_user_id'])
            for src, dst in [(u,v),(v,u)]:
                if dst in blacklist_set and src != dst:
                    neighbor_count[src] = neighbor_count.get(src,0) + 1
                    direct_neighbor[src] = 1

    # 來源②：crypto source_ip_hash（共用 IP）
    if 'source_ip_hash' in cs.columns:
        for ip, users in cs[cs['source_ip_hash'].notna()].groupby('source_ip_hash')['user_id'].apply(set).items():
            bl = users & blacklist_set
            for u in users:
                extra = bl - {u}
                if extra:
                    neighbor_count[u] = neighbor_count.get(u,0) + len(extra)
                    direct_neighbor[u] = 1

    # 來源③：twd source_ip_hash（TWD 共用 IP）
    ipc = 'source_ip_hash' if 'source_ip_hash' in twd.columns else (
          'source_ip'      if 'source_ip'      in twd.columns else None)
    if ipc:
        for ip, users in twd[twd[ipc].notna()].groupby(ipc)['user_id'].apply(set).items():
            bl = users & blacklist_set
            for u in users:
                extra = bl - {u}
                if extra:
                    neighbor_count[u] = neighbor_count.get(u,0) + len(extra)
                    direct_neighbor[u] = 1

    return pd.DataFrame({
        'user_id':                  list(all_user_ids),
        'blacklist_neighbor_count': [neighbor_count.get(u,0) for u in all_user_ids],
        'is_direct_neighbor':       [direct_neighbor.get(u,0) for u in all_user_ids],
    })

# ── 基本特徵（不含圖特徵）────────────────────────────────────────────────────
def build_base_features(train_label, predict_label):
    print('Loading tables...')
    user_info    = load_csv('user_info')
    twd          = load_csv('twd_transfer')
    crypto       = load_csv('crypto_transfer')
    usdt_trading = load_csv('usdt_twd_trading')

    all_users = pd.concat([
        train_label[['user_id']], predict_label[['user_id']]
    ]).drop_duplicates()

    # 1. user_info
    print('  [1/5] user_info...')
    ui = user_info.copy()
    ui['age'] = pd.to_numeric(ui.get('age', 0), errors='coerce').fillna(-1)
    ui['user_source'] = pd.to_numeric(ui.get('user_source', 0), errors='coerce').fillna(0)

    # kyc_level：明確判斷非空非"None"非"NaN"非"null"
    def is_done(col):
        if col not in ui.columns:
            return pd.Series(False, index=ui.index)
        s = ui[col].astype(str).str.strip()
        return ~s.isin(['', 'None', 'NaN', 'nan', 'null', 'nat', 'NaT'])

    l1 = is_done('level1_finished_at')
    l2 = is_done('level2_finished_at')
    ui['kyc_level'] = 0
    ui.loc[l1,       'kyc_level'] = 1
    ui.loc[l1 & l2,  'kyc_level'] = 2
    print(f'    kyc_level 分布: {ui["kyc_level"].value_counts().sort_index().to_dict()}')

    # 2. TWD
    print('  [2/5] twd_transfer...')
    twd['ori_samount'] = pd.to_numeric(twd.get('ori_samount', 0), errors='coerce').fillna(0)
    twd['kind']        = pd.to_numeric(twd.get('kind', 0),        errors='coerce').fillna(0)
    twd['created_at']  = pd.to_datetime(twd['created_at'], errors='coerce', utc=True)
    twd['hour']        = twd['created_at'].dt.hour
    twd_g = twd.groupby('user_id').agg(
        twd_deposit_count  = ('ori_samount', lambda x: (twd.loc[x.index,'kind']==0).sum()),
        twd_withdraw_count = ('ori_samount', lambda x: (twd.loc[x.index,'kind']==1).sum()),
        total_twd_volume   = ('ori_samount', 'sum'),
        avg_twd_amount     = ('ori_samount', 'mean'),
        night_tx_count     = ('hour', lambda x: ((x>=22)|(x<=5)).sum()),
        total_tx_count     = ('ori_samount', 'count'),
    ).reset_index()
    twd_g['night_tx_ratio'] = twd_g['night_tx_count'] / twd_g['total_tx_count'].replace(0,1)
    twd_s = twd.sort_values(['user_id','created_at'])
    twd_s['gap_sec'] = (twd_s['created_at'] - twd_s.groupby('user_id')['created_at'].shift(1)).dt.total_seconds()
    hs = twd_s[twd_s['gap_sec']<60].groupby('user_id').size().reset_index(name='high_speed_risk')
    twd_g = twd_g.merge(hs, on='user_id', how='left')
    twd_g['high_speed_risk'] = twd_g['high_speed_risk'].fillna(0)
    ip_col = 'source_ip_hash' if 'source_ip_hash' in twd.columns else (
             'source_ip'      if 'source_ip'      in twd.columns else None)
    if ip_col:
        ipd = twd[twd[ip_col].notna()]
        sh  = ipd.groupby(ip_col)['user_id'].nunique().reset_index(name='users_per_ip')
        ipd2 = ipd.merge(sh, on=ip_col)
        twd_g = twd_g.merge(ipd.groupby('user_id')[ip_col].nunique().reset_index(name='unique_ip_count'), on='user_id', how='left')
        twd_g = twd_g.merge(ipd2.groupby('user_id')['users_per_ip'].max().reset_index(name='max_ip_shared_users'), on='user_id', how='left')
        twd_g = twd_g.merge(ipd2[ipd2['users_per_ip']>5].groupby('user_id').size().reset_index(name='ip_anomaly'), on='user_id', how='left')
    else:
        twd_g['unique_ip_count'] = twd_g['max_ip_shared_users'] = twd_g['ip_anomaly'] = 0
    for c in ['unique_ip_count','max_ip_shared_users','ip_anomaly']:
        twd_g[c] = twd_g[c].fillna(0)

    # 3. Crypto
    print('  [3/5] crypto_transfer...')
    cs = crypto.copy()
    cs['kind']        = pd.to_numeric(cs.get('kind',0),        errors='coerce').fillna(0)
    cs['ori_samount'] = pd.to_numeric(cs.get('ori_samount',0), errors='coerce').fillna(0)
    cs['currency']    = cs.get('currency', pd.Series('', index=cs.index)).fillna('').astype(str)
    cs['created_at']  = pd.to_datetime(cs['created_at'], errors='coerce', utc=True)
    cs_g = cs.groupby('user_id').agg(
        crypto_deposit_count  = ('kind', lambda x: (x==0).sum()),
        crypto_withdraw_count = ('kind', lambda x: (x==1).sum()),
        crypto_currency_count = ('currency','nunique'),
    ).reset_index()
    cs_t = cs.groupby('user_id')['created_at'].agg(['min','max']).reset_index()
    cs_t.columns = ['user_id','first_tx','last_tx']
    cs_t['min_retention_minutes'] = ((cs_t['last_tx']-cs_t['first_tx']).dt.total_seconds()/60).fillna(0)
    cs_t['retention_event_count'] = cs.groupby('user_id').size().values
    cs_g = cs_g.merge(cs_t[['user_id','min_retention_minutes','retention_event_count']], on='user_id', how='left')

    # 4. USDT
    print('  [4/5] usdt_twd_trading...')
    ut = usdt_trading.copy()
    if 'trade_samount' in ut.columns and 'kind' in ut.columns:
        ut['trade_samount'] = pd.to_numeric(ut['trade_samount'], errors='coerce').fillna(0)
        ut['kind']          = pd.to_numeric(ut['kind'],          errors='coerce').fillna(0)
        ut_g = ut.groupby('user_id').agg(
            dep = ('trade_samount', lambda x: x[ut.loc[x.index,'kind']==0].sum()),
            wdw = ('trade_samount', lambda x: x[ut.loc[x.index,'kind']==1].sum()),
        ).reset_index()
        ut_g['asymmetry_flag'] = ((ut_g['dep']>0)&(ut_g['wdw']/ut_g['dep'].replace(0,1)>3)).astype(int)
    else:
        ut_g = ut[['user_id']].drop_duplicates() if 'user_id' in ut.columns else pd.DataFrame(columns=['user_id'])
        ut_g['asymmetry_flag'] = 0

    # 5. 合併（不含圖特徵）
    print('  [5/5] merging base features...')
    feat = all_users.merge(ui[['user_id','age','kyc_level','user_source']], on='user_id', how='left')
    feat = feat.merge(twd_g[['user_id','twd_deposit_count','twd_withdraw_count',
                              'total_twd_volume','avg_twd_amount','night_tx_ratio','high_speed_risk',
                              'unique_ip_count','max_ip_shared_users','ip_anomaly']], on='user_id', how='left')
    feat = feat.merge(cs_g[['user_id','crypto_deposit_count','crypto_withdraw_count',
                             'crypto_currency_count','min_retention_minutes','retention_event_count']], on='user_id', how='left')
    feat = feat.merge(ut_g[['user_id','asymmetry_flag']], on='user_id', how='left')
    feat = feat.fillna(0)
    print(f'  基本特徵：{feat.shape[0]:,} rows × {feat.shape[1]} cols')
    return feat, cs, twd

BASE_FEATS = [
    'age','kyc_level','user_source',
    'twd_deposit_count','twd_withdraw_count','total_twd_volume','avg_twd_amount',
    'night_tx_ratio','high_speed_risk',
    'crypto_deposit_count','crypto_withdraw_count','crypto_currency_count',
    'min_retention_minutes','retention_event_count',
    'unique_ip_count','max_ip_shared_users','ip_anomaly',
    'asymmetry_flag',
]
FEATS = BASE_FEATS  # 18 features，不含圖特徵（避免 fold 分布不一致）

LGB_PARAMS = dict(
    objective='binary', metric='auc', num_leaves=31, learning_rate=0.01,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    min_child_samples=20, seed=42, verbosity=-1, n_jobs=-1,
)
NUM_ROUNDS = 500

# ════════════════════════════════════════════════════════════
print('=' * 55)
print('  BitoGuard AML — LightGBM 訓練（Fold-Aware 圖特徵）')
print('=' * 55)

train_label   = load_csv('train_label')
predict_label = load_csv('predict_label')
train_label['status'] = pd.to_numeric(train_label['status'], errors='coerce').fillna(0).astype(int)
print(f'訓練集：{len(train_label):,}  黑名單：{train_label["status"].sum():,}')
print(f'預測集：{len(predict_label):,}')

# 計算基本特徵
base_feat, cs_df, twd_df = build_base_features(train_label, predict_label)

train_ids = set(train_label['user_id'])
pred_ids  = set(predict_label['user_id'])
all_ids   = list(base_feat['user_id'])

train_base = base_feat[base_feat['user_id'].isin(train_ids)].merge(
    train_label[['user_id','status']], on='user_id', how='left')
pred_base  = base_feat[base_feat['user_id'].isin(pred_ids)]

X_base      = train_base[BASE_FEATS].values.astype(np.float32)
y           = train_base['status'].values.astype(int)
X_pred_base = pred_base[BASE_FEATS].values.astype(np.float32)
pred_user_ids = pred_base['user_id'].values

print(f'\n基本特徵矩陣：{X_base.shape}  正類比率：{y.mean():.4f}')

# ── Fold-Aware 5-Fold CV ─────────────────────────────────────────────────────
print('\n[Stage 2] Fold-Aware 5-Fold StratifiedKFold...')
skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs  = np.zeros(len(y),      dtype=np.float32)
test_probs = np.zeros(len(X_pred_base), dtype=np.float32)
fold_results = []
train_user_ids = train_base['user_id'].values

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_base, y), 1):
    # Fold-aware blacklist：只用訓練折的黑名單（無洩漏）
    fold_blacklist = set(train_user_ids[tr_idx][y[tr_idx] == 1])

    # 為所有用戶計算圖特徵（用 fold-specific blacklist）
    graph_all = compute_graph_features(all_ids, cs_df, twd_df, fold_blacklist)
    graph_all = graph_all.set_index('user_id')

    # 組合訓練折特徵
    tr_uids  = train_user_ids[tr_idx]
    val_uids = train_user_ids[val_idx]
    X_tr  = X_base[tr_idx].copy()
    X_val = X_base[val_idx].copy()
    y_tr, y_val = y[tr_idx], y[val_idx]

    dtrain = lgb.Dataset(X_tr,  label=y_tr,  feature_name=FEATS)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=FEATS, reference=dtrain)
    model  = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_ROUNDS,
                       valid_sets=[dval], callbacks=[lgb.log_evaluation(period=-1)])

    val_prob  = model.predict(X_val)
    # 預測集使用 fold blacklist（保持一致性）
    test_prob    = model.predict(X_pred_base)

    oof_probs[val_idx] = val_prob
    test_probs += test_prob / 5

    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.03, 0.85, 0.01):
        f = f1_score(y_val, (val_prob>=t).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_t = f, t
    auc  = roc_auc_score(y_val, val_prob)
    prec = precision_score(y_val, (val_prob>=best_t).astype(int), zero_division=0)
    rec  = recall_score(y_val,   (val_prob>=best_t).astype(int), zero_division=0)
    fold_results.append({'fold':fold,'threshold':round(float(best_t),2),
                         'precision':round(float(prec),4),'recall':round(float(rec),4),
                         'f1':round(float(best_f1),4),'auc':round(float(auc),4),
                         'val_pos':int(y_val.sum())})
    if fold == 5:
        fi = pd.DataFrame({'feature':FEATS, 'importance':model.feature_importance(importance_type='gain')})
        fi.sort_values('importance',ascending=False).to_csv(BASE/'feature_importance.csv',index=False)
    print(f'  Fold {fold}: AUC={auc:.4f}  F1={best_f1:.4f}  P={prec:.4f}  R={rec:.4f}  thr={best_t:.2f}  '
          f'fold_bl={len(fold_blacklist)}')

# ── OOF 全局 ─────────────────────────────────────────────────────────────────
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.03, 0.85, 0.01):
    f = f1_score(y, (oof_probs>=t).astype(int), zero_division=0)
    if f > best_f1: best_f1, best_t = f, t
oof_pred = (oof_probs >= best_t).astype(int)
oof_prec = precision_score(y, oof_pred, zero_division=0)
oof_rec  = recall_score(y,   oof_pred, zero_division=0)
oof_auc  = roc_auc_score(y,  oof_probs)
tn, fp, fn, tp = confusion_matrix(y, oof_pred, labels=[0,1]).ravel()

pd.DataFrame({'true_label':y,'oof_prob':oof_probs}).to_csv(BASE/'oof_predictions.csv',index=False)

# ── 生成提交（用完整訓練黑名單計算最終圖特徵）─────────────────────────────────
print('\n[Stage 3] 生成 submission（全量黑名單圖特徵）...')
full_blacklist = set(train_label.loc[train_label['status']==1, 'user_id'])
graph_final    = compute_graph_features(all_ids, cs_df, twd_df, full_blacklist).set_index('user_id')
X_pred_final   = np.hstack([X_pred_base,
                              graph_final.loc[pred_user_ids, ['blacklist_neighbor_count','is_direct_neighbor']].values])
# 用最後一折模型預測（或平均）
pred_labels = (test_probs >= best_t).astype(int)
sub = pd.DataFrame({'user_id': pred_user_ids, 'status': pred_labels})
sub.to_csv(BASE/'submission.csv', index=False)
sub_prob = sub.copy(); sub_prob['probability'] = test_probs
sub_prob.to_csv(BASE/'submission_with_prob.csv', index=False)

report = {
    'model':'LightGBM','params':LGB_PARAMS,'num_rounds':NUM_ROUNDS,'features':FEATS,
    'oof_metrics':{'auc':round(float(oof_auc),4),'f1':round(float(best_f1),4),
                   'precision':round(float(oof_prec),4),'recall':round(float(oof_rec),4),
                   'threshold':round(float(best_t),2)},
    'submission':{'total':int(len(sub)),'blacklist':int(pred_labels.sum()),
                  'normal':int((pred_labels==0).sum())},
    'folds':fold_results,
}
with open(BASE/'cv_report_lgb.json','w',encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f'\n{"="*55}')
print('  ★ 判斷成功率 (OOF Fold-Aware) ★')
print(f'{"="*55}')
print(f'  AUC      : {oof_auc:.4f}')
print(f'  F1-Score : {best_f1:.4f}')
print(f'  Precision: {oof_prec:.4f}  ({oof_prec*100:.1f}% 預測為黑名單的是真的)')
print(f'  Recall   : {oof_rec:.4f}  (抓到 {oof_rec*100:.1f}% 實際黑名單)')
print(f'  門檻值   : {best_t:.2f}')
print(f'  混淆矩陣  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}')
print(f'  黑名單預測: {pred_labels.sum():,} / {len(sub):,}')
print(f'{"="*55}')

In [ ]:
# ── Cell 4：視覺化結果 ───────────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

# 混淆矩陣
fig_cm = go.Figure(go.Heatmap(
    z=[[tn, fp],[fn, tp]],
    x=['預測：正常(0)','預測：黑名單(1)'],
    y=['實際：正常(0)','實際：黑名單(1)'],
    text=[[f'TN\n{tn:,}',f'FP\n{fp:,}'],[f'FN\n{fn:,}',f'TP\n{tp:,}']],
    texttemplate='%{text}', textfont={'size':18,'color':'white'},
    colorscale=[[0,'#27ae60'],[0.5,'#f39c12'],[1,'#e74c3c']], showscale=False,
))
fig_cm.update_layout(title='混淆矩陣（OOF 5-Fold）', height=320, width=450)
fig_cm.show()

# 門檻掃描曲線
ts = np.arange(0.03, 0.85, 0.01)
fs = [f1_score(y, (oof_probs>=t).astype(int), zero_division=0) for t in ts]
ps = [precision_score(y, (oof_probs>=t).astype(int), zero_division=0) for t in ts]
rs = [recall_score(y, (oof_probs>=t).astype(int), zero_division=0) for t in ts]
fig_thr = go.Figure()
fig_thr.add_trace(go.Scatter(x=ts, y=fs, name='F1',       line=dict(color='#27ae60', width=2.5)))
fig_thr.add_trace(go.Scatter(x=ts, y=ps, name='Precision', line=dict(color='#2980b9', width=2)))
fig_thr.add_trace(go.Scatter(x=ts, y=rs, name='Recall',    line=dict(color='#e74c3c', width=2)))
fig_thr.add_vline(x=best_t, line_dash='dash', line_color='gold', annotation_text=f'最佳門檻={best_t:.2f}')
fig_thr.update_layout(title='門檻值 vs Precision / Recall / F1', xaxis_title='門檻值',
                      yaxis_range=[0,1], height=350, legend=dict(orientation='h',y=-0.25))
fig_thr.show()

# 特徵重要度
fi_df = pd.read_csv(BASE / 'feature_importance.csv').head(15)
fig_fi = px.bar(fi_df, x='importance', y='feature', orientation='h',
                color='importance', color_continuous_scale='RdYlGn_r',
                title='Top-15 特徵重要度（LightGBM Gain）')
fig_fi.update_yaxes(autorange='reversed')
fig_fi.update_layout(height=450, coloraxis_showscale=False)
fig_fi.show()

# 預測機率分布
sub_prob = pd.read_csv(BASE / 'submission_with_prob.csv')
fig_dist = px.histogram(sub_prob, x='probability', color='status', nbins=80,
                        barmode='overlay', opacity=0.75,
                        color_discrete_map={0:'#4C72B0',1:'#C44E52'},
                        title='預測機率分布（藍=正常 / 紅=黑名單）')
fig_dist.add_vline(x=best_t, line_dash='dash', line_color='gold',
                   annotation_text=f'門檻 {best_t:.2f}')
fig_dist.update_layout(height=300)
fig_dist.show()

print('圖表生成完成')

In [ ]:
# ── Cell 5：修正並啟動 Streamlit ─────────────────────────────────────────────
import subprocess, json, time
from pathlib import Path

BASE = Path('/home/ec2-user/SageMaker')

# ── 寫入修正後的 app.py（每次執行都覆蓋，確保最新版）────────────────────────
app_code = '''
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
warnings.filterwarnings("ignore")

BASE = Path("/home/ec2-user/SageMaker")
st.set_page_config(page_title="BitoGuard AML", page_icon="🛡️", layout="wide")

@st.cache_data
def load_report():
    with open(BASE / "cv_report_lgb.json", encoding="utf-8") as f:
        return json.load(f)

@st.cache_data
def load_sub():
    return pd.read_csv(BASE / "submission_with_prob.csv")

@st.cache_data
def load_fi():
    p = BASE / "feature_importance.csv"
    return pd.read_csv(p) if p.exists() else None

@st.cache_data
def load_oof():
    p = BASE / "oof_predictions.csv"
    return pd.read_csv(p) if p.exists() else None

r   = load_report()
m   = r["oof_metrics"]
s   = r["submission"]
sub = load_sub()
fi  = load_fi()
oof = load_oof()
folds = r["folds"]

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### 🛡️ BitoGuard AML")
    st.markdown("---")
    page = st.radio("頁面", ["📊 總覽", "🔍 用戶查詢", "📈 模型評估", "📋 提交結果"])
    st.markdown("---")
    st.markdown(f"**模型：** LightGBM")
    st.markdown(f"**OOF F1：** `{m[\'f1\']:.4f}`")
    st.markdown(f"**AUC：** `{m[\'auc\']:.4f}`")
    st.markdown(f"**黑名單：** `{s[\'blacklist\']:,}` / `{s[\'total\']:,}`")
    if st.button("🔄 重新整理"):
        st.cache_data.clear()
        st.rerun()

# ── 頁面 1：總覽 ──────────────────────────────────────────────────────────────
if page == "📊 總覽":
    st.title("🛡️ BitoGuard AML 反洗錢偵測系統")
    c1,c2,c3,c4,c5 = st.columns(5)
    c1.metric("AUC",       f"{m[\'auc\']:.4f}", help="ROC 曲線下面積")
    c2.metric("F1-Score",  f"{m[\'f1\']:.4f}")
    c3.metric("Precision", f"{m[\'precision\']:.4f}", help="預測黑名單中是真的比例")
    c4.metric("Recall",    f"{m[\'recall\']:.4f}",    help="實際黑名單被抓到的比例")
    c5.metric("黑名單預測", f"{s[\'blacklist\']:,}",  delta=f"/ {s[\'total\']:,} 用戶")
    st.markdown("---")

    col_l, col_r = st.columns(2)
    with col_l:
        st.subheader("混淆矩陣 (OOF 5-Fold)")
        N_POS = 1640
        tp_v = int(m["recall"] * N_POS)
        fp_v = max(0, int(tp_v / m["precision"] - tp_v)) if m["precision"] > 0 else 0
        fn_v = N_POS - tp_v
        tn_v = 51017 - N_POS - fp_v
        cm_fig = go.Figure(go.Heatmap(
            z=[[tn_v, fp_v],[fn_v, tp_v]],
            x=["預測：正常(0)","預測：黑名單(1)"],
            y=["實際：正常(0)","實際：黑名單(1)"],
            text=[[f"TN\\n{tn_v:,}",f"FP\\n{fp_v:,}"],[f"FN\\n{fn_v:,}",f"TP\\n{tp_v:,}"]],
            texttemplate="%{text}", textfont={"size":18,"color":"white"},
            colorscale=[[0,"#27ae60"],[0.5,"#f39c12"],[1,"#e74c3c"]], showscale=False,
        ))
        cm_fig.update_layout(height=320, margin=dict(l=0,r=0,t=10,b=0))
        st.plotly_chart(cm_fig, use_container_width=True)

    with col_r:
        st.subheader("5-Fold 交叉驗證")
        xs = [f"Fold {f[\'fold\']}" for f in folds]
        fold_fig = go.Figure()
        fold_fig.add_trace(go.Bar(name="Precision", x=xs, y=[f["precision"] for f in folds], marker_color="#4C72B0"))
        fold_fig.add_trace(go.Bar(name="Recall",    x=xs, y=[f["recall"]    for f in folds], marker_color="#DD8452"))
        fold_fig.add_trace(go.Bar(name="F1-Score",  x=xs, y=[f["f1"]        for f in folds], marker_color="#55A868"))
        fold_fig.add_hline(y=m["f1"], line_dash="dash", line_color="#55A868",
                           annotation_text=f"OOF F1={m[\'f1\']:.4f}")
        fold_fig.update_layout(barmode="group", height=320, yaxis_range=[0,1],
                               margin=dict(l=0,r=0,t=10,b=0))
        st.plotly_chart(fold_fig, use_container_width=True)

# ── 頁面 2：用戶查詢 ──────────────────────────────────────────────────────────
elif page == "🔍 用戶查詢":
    st.title("🔍 用戶風險查詢")
    uid_input = st.text_input("輸入用戶 ID", placeholder="例如：967903")
    if uid_input:
        try:
            uid = int(uid_input.strip())
            row = sub[sub["user_id"]==uid]
            if row.empty:
                st.warning(f"找不到 user_id={uid}")
            else:
                row = row.iloc[0]
                prob   = float(row.get("probability", 0.5))
                status = int(row["status"])
                color  = "#c0392b" if status==1 else "#27ae60"
                label  = "🔴 黑名單" if status==1 else "🟢 正常"
                st.markdown(f"""<div style="border:3px solid {color};border-radius:12px;padding:20px">
                    <h2 style="color:{color}">用戶 {uid} — {label}</h2>
                    <p>預測機率：<b style="font-size:24px;color:{color}">{prob:.4f}</b>　門檻：{m["threshold"]:.2f}</p>
                </div>""", unsafe_allow_html=True)
                gauge = go.Figure(go.Indicator(mode="gauge+number", value=prob*100,
                    title={"text":"風險分數 (%)"},
                    gauge={"axis":{"range":[0,100]},"bar":{"color":color},
                           "threshold":{"line":{"color":"black","width":3},"value":m["threshold"]*100}}))
                gauge.update_layout(height=280, margin=dict(l=20,r=20,t=30,b=0))
                st.plotly_chart(gauge, use_container_width=True)
        except ValueError:
            st.error("請輸入數字")
    st.subheader("高風險用戶 Top 50")
    if "probability" in sub.columns:
        top50 = sub.nlargest(50,"probability")[["user_id","probability","status"]]
        st.dataframe(top50.style.background_gradient(subset=["probability"],cmap="Reds"), use_container_width=True)

# ── 頁面 3：模型評估 ──────────────────────────────────────────────────────────
elif page == "📈 模型評估":
    st.title("📈 模型評估")
    col_l, col_r = st.columns(2)
    with col_l:
        st.subheader("特徵重要度")
        if fi is not None:
            fig = px.bar(fi.head(15), x="importance", y="feature", orientation="h",
                         color="importance", color_continuous_scale="RdYlGn_r")
            fig.update_yaxes(autorange="reversed")
            fig.update_layout(height=480, coloraxis_showscale=False, margin=dict(l=0,r=0,t=0,b=0))
            st.plotly_chart(fig, use_container_width=True)
    with col_r:
        st.subheader("門檻值分析")
        if oof is not None:
            from sklearn.metrics import f1_score, precision_score, recall_score
            y_t = oof["true_label"].values; y_p = oof["oof_prob"].values
            ts  = np.arange(0.03,0.85,0.01)
            fs  = [f1_score(y_t,(y_p>=t).astype(int),zero_division=0) for t in ts]
            ps  = [precision_score(y_t,(y_p>=t).astype(int),zero_division=0) for t in ts]
            rs  = [recall_score(y_t,(y_p>=t).astype(int),zero_division=0) for t in ts]
            fig2 = go.Figure()
            fig2.add_trace(go.Scatter(x=ts,y=fs,name="F1",       line=dict(color="#27ae60",width=2.5)))
            fig2.add_trace(go.Scatter(x=ts,y=ps,name="Precision",line=dict(color="#2980b9",width=2)))
            fig2.add_trace(go.Scatter(x=ts,y=rs,name="Recall",   line=dict(color="#e74c3c",width=2)))
            fig2.add_vline(x=m["threshold"],line_dash="dash",line_color="gold",
                           annotation_text=f"門檻={m[\'threshold\']:.2f}")
            fig2.update_layout(height=280, yaxis_range=[0,1], margin=dict(l=0,r=0,t=10,b=0))
            st.plotly_chart(fig2, use_container_width=True)
        st.subheader("各 Fold 指標")
        df_f = pd.DataFrame(folds)
        df_f.columns = ["Fold","門檻","Precision","Recall","F1","AUC","正類數"]
        st.dataframe(df_f.style.highlight_max(subset=["F1","AUC"],color="#d5f5e3")
                              .format({"Precision":"{:.4f}","Recall":"{:.4f}","F1":"{:.4f}","AUC":"{:.4f}"}),
                     use_container_width=True)
    if "probability" in sub.columns:
        fig3 = px.histogram(sub, x="probability", color="status", nbins=80,
                            barmode="overlay", opacity=0.75,
                            color_discrete_map={0:"#4C72B0",1:"#C44E52"},
                            title="預測機率分布")
        fig3.add_vline(x=m["threshold"],line_dash="dash",line_color="gold",
                       annotation_text=f"門檻 {m[\'threshold\']:.2f}")
        fig3.update_layout(height=280)
        st.plotly_chart(fig3, use_container_width=True)

# ── 頁面 4：提交結果 ──────────────────────────────────────────────────────────
elif page == "📋 提交結果":
    st.title("📋 提交結果")
    c1,c2,c3 = st.columns(3)
    c1.metric("總用戶數",  f"{s[\'total\']:,}")
    c2.metric("預測黑名單", f"{s[\'blacklist\']:,}", delta=f"{s[\'blacklist\']/s[\'total\']*100:.2f}%")
    c3.metric("預測正常",  f"{s[\'normal\']:,}")
    pie = px.pie(values=[s["blacklist"],s["normal"]],names=["黑名單(1)","正常(0)"],
                 color_discrete_sequence=["#e74c3c","#27ae60"],hole=0.45)
    pie.update_layout(height=300, margin=dict(l=0,r=0,t=20,b=0))
    st.plotly_chart(pie, use_container_width=True)
    fopt = st.selectbox("篩選",["全部","只看黑名單","只看正常"])
    show = sub.copy()
    if fopt=="只看黑名單": show=show[show["status"]==1]
    elif fopt=="只看正常":  show=show[show["status"]==0]
    st.dataframe(show, use_container_width=True, height=380)
    st.download_button("⬇️ 下載 submission.csv",
        sub[["user_id","status"]].to_csv(index=False).encode(),
        "submission.csv","text/csv", use_container_width=True)
'''

(BASE / 'app.py').write_text(app_code, encoding='utf-8')
print('app.py 已更新')

# ── 取得正確的 region + notebook 名稱 ────────────────────────────────────────
try:
    import boto3
    region = boto3.session.Session().region_name or 'us-west-2'
except Exception:
    region = 'us-west-2'

try:
    import json
    meta = Path('/opt/ml/metadata/resource-metadata.json')
    info = json.loads(meta.read_text())
    nb_name = info.get('ResourceName', '')
except Exception:
    nb_name = ''

proxy_url = f'https://{nb_name}.notebook.{region}.sagemaker.aws/proxy/8501/'

# ── 停掉舊的，重新啟動 ───────────────────────────────────────────────────────
import subprocess, time
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

proc = subprocess.Popen([
    'python', '-m', 'streamlit', 'run', str(BASE / 'app.py'),
    '--server.port',                 '8501',
    '--server.address',              '0.0.0.0',
    '--server.headless',             'true',
    '--server.enableCORS',           'false',
    '--server.enableXsrfProtection', 'false',
    '--browser.gatherUsageStats',    'false',
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(4)

if proc.poll() is None:
    print(f'Streamlit 啟動成功！')
    print(f'\n開啟瀏覽器：\n  {proxy_url}')
    print(f'\nNotebook: {nb_name}  Region: {region}')
else:
    print('啟動失敗：', proc.stderr.read().decode()[-300:])